In [5]:
import requests
import json

In [6]:
r = requests.post(
    'https://api.semanticscholar.org/graph/v1/paper/batch',
    params={'fields': 'referenceCount,citationCount,title'},
    json={"ids": ["649def34f8be52c8b66281af98ae884c09aef38b", "ARXIV:2106.15928"]} # gets specific id
)

In [7]:
print(json.dumps(r.json(), indent=2))

[
  {
    "paperId": "649def34f8be52c8b66281af98ae884c09aef38b",
    "title": "Construction of the Literature Graph in Semantic Scholar",
    "referenceCount": 26,
    "citationCount": 383
  },
  {
    "paperId": "f712fab0d58ae6492e3cdfc1933dae103ec12d5d",
    "title": "Reinfection and low cross-immunity as drivers of epidemic resurgence under high seroprevalence: a model-based approach with application to Amazonas, Brazil",
    "referenceCount": 13,
    "citationCount": 0
  }
]


In [1]:
import requests
import networkx as nx
import json
from typing import Dict, List, Optional
import time
from datetime import datetime, timedelta

class AcademicKnowledgeGraph:
    def __init__(self, api_key: Optional[str] = None, max_papers: int = 20):
        self.base_url = "https://api.semanticscholar.org/graph/v1"
        self.headers = {"x-api-key": api_key} if api_key else {}
        self.graph = nx.DiGraph()
        self.last_request_time = datetime.now()
        self.request_interval = 0.2 if api_key else 1.0
        self.max_papers = max_papers
        self.paper_count = 0
        
    def _wait_for_rate_limit(self):
        now = datetime.now()
        elapsed = now - self.last_request_time
        if elapsed.total_seconds() < self.request_interval:
            time.sleep(self.request_interval - elapsed.total_seconds())
        self.last_request_time = datetime.now()

    def _make_request(self, endpoint: str, params: Dict = None, max_retries: int = 3) -> Dict:
        for attempt in range(max_retries):
            self._wait_for_rate_limit()
            
            try:
                response = requests.get(
                    f"{self.base_url}/{endpoint}", 
                    headers=self.headers,
                    params=params
                )
                
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 429:
                    wait_time = 2 ** attempt
                    print(f"Rate limit hit, waiting {wait_time} seconds...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"Error making request: {response.status_code}")
                    print(f"Response: {response.text}")
                    break
                    
            except Exception as e:
                print(f"Request failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                break
                
        return None

    def _clean_attributes(self, attrs):
        cleaned = {}
        for k, v in attrs.items():
            if isinstance(v, (list, dict)):
                cleaned[k] = json.dumps(v)
            elif v is None:
                cleaned[k] = ""
            elif isinstance(v, (str, int, float, bool)):
                cleaned[k] = v
            else:
                cleaned[k] = str(v)
        return cleaned

    def build_from_paper(self, 
                        paper_identifier: str,
                        max_citations_per_paper: int = 5,
                        min_citations: int = 0,
                        fields_of_study: List[str] = None) -> nx.DiGraph:
        """
        Build knowledge graph starting from a paper
        
        Args:
            paper_identifier: DOI or title of the paper
            max_citations_per_paper: Maximum number of citations to include per paper
            min_citations: Minimum citation count for included papers
            fields_of_study: List of fields to filter by
        """
        print(f"Searching for paper: {paper_identifier}")
        
        # Check if we've reached the maximum number of papers
        if self.paper_count >= self.max_papers:
            print("Reached maximum number of papers")
            return self.graph
        
        # Get initial paper
        if paper_identifier.startswith("DOI:"):
            paper = self._make_request(f"paper/{paper_identifier}", {
                "fields": "paperId,title,abstract,authors,citations,references,fieldsOfStudy,venue,year,tldr"
            })
        else:
            search_result = self._make_request("paper/search", {
                "query": paper_identifier,
                "limit": 1,
                "fields": "paperId,title,abstract,authors,citations,references,fieldsOfStudy,venue,year,tldr"
            })
            
            if search_result and search_result.get("data"):
                paper = search_result["data"][0]
                print(f"Found paper: {paper.get('title')}")
            else:
                paper = None

        if not paper:
            print(f"Could not find paper: {paper_identifier}")
            return None
            
        # Skip if paper already in graph
        if paper["paperId"] in self.graph:
            return self.graph
            
        # Add initial paper to graph
        paper_attrs = {
            "type": "paper",
            "title": paper.get("title", ""),
            "abstract": paper.get("abstract", ""),
            "year": paper.get("year", ""),
            "venue": paper.get("venue", ""),
            "fields": json.dumps(paper.get("fieldsOfStudy", []))
        }
        self.graph.add_node(paper["paperId"], **self._clean_attributes(paper_attrs))
        self.paper_count += 1
        print(f"Added paper {self.paper_count}/{self.max_papers}: {paper.get('title')}")
                           
        # Add authors (but don't count toward paper limit)
        if "authors" in paper:
            for author in paper["authors"]:
                author_attrs = {
                    "type": "author",
                    "name": author.get("name", "")
                }
                self.graph.add_node(author["authorId"], **self._clean_attributes(author_attrs))
                self.graph.add_edge(author["authorId"], 
                                  paper["paperId"],
                                  type="authored")
            print(f"Added {len(paper['authors'])} authors")
                                  
        # Get citations
        citations = self._make_request(f"paper/{paper['paperId']}/citations", {
            "fields": "paperId,title,abstract,year,citationCount,fieldsOfStudy,contexts,isInfluential",
            "limit": 100
        })
        
        if citations and "data" in citations:
            # Sort citations by influence and citation count
            sorted_citations = sorted(
                citations["data"],
                key=lambda x: (x.get("isInfluential", False), x.get("citationCount", 0)),
                reverse=True
            )
            
            # Take only the top N most relevant citations
            relevant_citations = sorted_citations[:max_citations_per_paper]
            
            added_citations = 0
            for citation in relevant_citations:
                # Check maximum papers limit
                if self.paper_count >= self.max_papers:
                    print("Reached maximum number of papers")
                    return self.graph
                    
                if (min_citations and 
                    citation.get("citationCount", 0) < min_citations):
                    continue
                    
                if (fields_of_study and 
                    not any(field in citation.get("fieldsOfStudy", [])
                           for field in fields_of_study)):
                    continue
                
                # Add citation to graph
                citation_attrs = {
                    "type": "paper",
                    "title": citation.get("title", ""),
                    "abstract": citation.get("abstract", ""),
                    "year": citation.get("year", ""),
                    "fields": json.dumps(citation.get("fieldsOfStudy", []))
                }
                self.graph.add_node(citation["paperId"], **self._clean_attributes(citation_attrs))
                self.paper_count += 1
                
                edge_attrs = {
                    "type": "cites",
                    "contexts": json.dumps(citation.get("contexts", [])),
                    "is_influential": citation.get("isInfluential", False)
                }
                self.graph.add_edge(citation["paperId"],
                                  paper["paperId"],
                                  **self._clean_attributes(edge_attrs))
                added_citations += 1
                
                # Recursively add citations from this paper
                self.build_from_paper(
                    citation["paperId"],
                    max_citations_per_paper=max_citations_per_paper,
                    min_citations=min_citations,
                    fields_of_study=fields_of_study
                )
                    
            print(f"Added {added_citations} citations")
                                        
        return self.graph

    def export_to_file(self, filename: str, format: str = "json"):
        print(f"\nExporting graph with {self.graph.number_of_nodes()} nodes and {self.graph.number_of_edges()} edges")
        
        if format == "json":
            data = {
                "nodes": [{"id": n, **self.graph.nodes[n]} for n in self.graph.nodes()],
                "edges": [{"source": u, "target": v, **self.graph.edges[u, v]} 
                         for u, v in self.graph.edges()]
            }
            with open(filename, 'w') as f:
                json.dump(data, f, indent=2)
        
        elif format == "graphml":
            nx.write_graphml(self.graph, filename)
            
        print(f"Graph exported to {filename}")

# Example usage with limits:
if __name__ == "__main__":
    # Get API key from environment or user input
    api_key = input("Enter your Semantic Scholar API key (press Enter if none): ").strip() or None
    
    # Initialize with API key and max papers limit
    kg_builder = AcademicKnowledgeGraph(api_key=api_key, max_papers=20)
    
    # Build graph with limited citations per paper
    graph = kg_builder.build_from_paper(
        "Construction of the Literature Graph in Semantic Scholar",
        max_citations_per_paper=5,  # Only get top 5 citations per paper
        min_citations=50,  # Papers must have at least 50 citations
        fields_of_study=["Computer Science"]
    )
    
    # Export
    kg_builder.export_to_file("academic_knowledge_graph.json", format="json")

Searching for paper: Construction of the Literature Graph in Semantic Scholar
Rate limit hit, waiting 1 seconds...
Rate limit hit, waiting 2 seconds...
Found paper: Construction of the Literature Graph in Semantic Scholar
Added paper 1/20: Construction of the Literature Graph in Semantic Scholar
Added 23 authors
Added 0 citations

Exporting graph with 24 nodes and 23 edges
Graph exported to academic_knowledge_graph.json
